# Credit Scoring — Practical Interview Prep Notebook

Hands-on exercises built around the columns in `final_enriched_dataset.csv`.
Run each section, read the commentary, and reproduce the outputs from scratch.

**Sections:**
1. Data Loading & Inspection
2. Bad Rate Analysis (SQL-equivalent in Python)
3. Feature Engineering
4. WoE / IV Calculation
5. PSI — Population Stability Index
6. Calibration Curve
7. Scorecard Cut-off Strategy
8. Vintage / Cohort Analysis
9. SHAP Explainability
10. End-to-end Mini Pipeline (from raw to scored)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score, balanced_accuracy_score,
    confusion_matrix, roc_curve
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from xgboost import XGBClassifier

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 50)

print('Libraries loaded.')

---
## 1. Data Loading & Inspection

**Key interview concepts:**
- Bad rate (class balance)
- Missing value analysis
- Feature type identification (categorical vs numerical)

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────
# Try both possible filenames
import os
for fname in ['final_enriched_dataset.csv', 'final_enriched_output.csv']:
    if os.path.exists(fname):
        df_raw = pd.read_csv(fname)
        print(f'Loaded: {fname}')
        break
else:
    # Create a synthetic dataset that mimics the schema for practice
    np.random.seed(42)
    n = 5000
    df_raw = pd.DataFrame({
        'cedula':              range(n),
        'target':              np.random.choice([0, 1], n, p=[0.85, 0.15]),
        'experian_score':      np.random.normal(620, 80, n).clip(300, 900),
        'queries_6m':          np.random.poisson(2, n),
        'queries_12m':         np.random.poisson(4, n),
        'active_credits':      np.random.poisson(3, n),
        'creditos_mora':       np.random.choice([0, 1, 2], n, p=[0.7, 0.2, 0.1]),
        'total_debt':          np.random.exponential(5_000_000, n),
        'total_arrears_balance': np.random.exponential(200_000, n) * np.random.choice([0, 1], n, p=[0.8, 0.2]),
        'l_principal':         np.random.uniform(500_000, 10_000_000, n),
        'l_term':              np.random.choice([6, 12, 18, 24, 36], n),
        'edad':                np.random.normal(38, 10, n).clip(18, 70),
        'ingresos_smlv':       np.random.exponential(5, n).clip(1, 30),
        'cuota_mensual':       np.random.uniform(50_000, 800_000, n),
        'departamento':        np.random.choice(['BOGOTA', 'ANTIOQUIA', 'VALLE', 'ATLANTICO', 'SANTANDER'], n),
        'genero':              np.random.choice(['M', 'F'], n),
        'hist_neg_12m':        np.random.choice([0, 1, 2, 3], n, p=[0.65, 0.2, 0.1, 0.05]),
        'platam_score':        np.random.normal(500, 100, n).clip(200, 800),
        'hybrid_score':        np.random.normal(510, 95, n).clip(200, 800),
    })
    # Introduce some NULLs
    for col in ['experian_score', 'queries_6m', 'total_arrears_balance']:
        mask = np.random.rand(n) < 0.08
        df_raw.loc[mask, col] = np.nan
    print('Created synthetic dataset (final_enriched_dataset.csv not found).')
    df_raw.to_csv('final_enriched_dataset.csv', index=False)

# Rename target if needed
if 'target_dpd0' in df_raw.columns and 'target' not in df_raw.columns:
    df_raw.rename(columns={'target_dpd0': 'target'}, inplace=True)

print(f'Shape: {df_raw.shape}')
print(f'\nTarget distribution:')
print(df_raw['target'].value_counts(normalize=True).rename({0: 'Good (0)', 1: 'Bad (1)'}))

In [ ]:
# ── Null Analysis ─────────────────────────────────────────────────────
null_pct = (df_raw.isnull().sum() / len(df_raw) * 100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]

print('Columns with missing values:')
print(null_pct.to_string())

# Columns to drop (>40% null — as per pipeline threshold)
drop_threshold = 40
to_drop = null_pct[null_pct > drop_threshold].index.tolist()
print(f'\nColumns exceeding {drop_threshold}% null threshold: {to_drop}')

---
## 2. Bad Rate Analysis by Feature Band

**Interview concept:** This is the credit-scoring equivalent of EDA.
Before modelling, you need to understand how bad rate varies across each feature.
This is also how you manually compute WoE/IV.

In [ ]:
def bad_rate_by_band(df, feature, target='target', n_bins=5, is_cat=False):
    """
    Compute bad rate, goods, bads, and WoE per band of a feature.
    For numerical: bins with quantile-based edges.
    For categorical: group by category value.
    """
    df = df[[feature, target]].dropna()

    if is_cat:
        grouped = df.groupby(feature)[target]
    else:
        df['_band'] = pd.qcut(df[feature], q=n_bins, duplicates='drop')
        grouped = df.groupby('_band')[target]

    stats = grouped.agg(['sum', 'count']).rename(columns={'sum': 'bads', 'count': 'total'})
    stats['goods'] = stats['total'] - stats['bads']
    stats['bad_rate'] = stats['bads'] / stats['total']

    total_bads  = stats['bads'].sum()
    total_goods = stats['goods'].sum()
    stats['pct_bads']  = stats['bads']  / total_bads
    stats['pct_goods'] = stats['goods'] / total_goods
    stats['woe'] = np.log(
        stats['pct_goods'].clip(lower=1e-8) / stats['pct_bads'].clip(lower=1e-8)
    )
    stats['iv'] = (stats['pct_goods'] - stats['pct_bads']) * stats['woe']

    return stats


# ── Bad rate by Experian score band ───────────────────────────────────
stats_exp = bad_rate_by_band(df_raw, 'experian_score', n_bins=5)
print('=== Experian Score — Bad Rate by Band ===')
print(stats_exp[['total', 'bads', 'goods', 'bad_rate', 'woe', 'iv']].to_string())
print(f'\nTotal IV (experian_score): {stats_exp["iv"].sum():.4f}')

In [ ]:
# ── IV Ranking across all numeric features ────────────────────────────
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['target', 'cedula']]

iv_results = {}
for col in numeric_cols:
    try:
        s = bad_rate_by_band(df_raw, col, n_bins=5)
        iv_results[col] = s['iv'].sum()
    except Exception:
        iv_results[col] = np.nan

iv_series = pd.Series(iv_results).sort_values(ascending=False)
print('=== IV Ranking ===')
print(iv_series.to_string())

# Visualise
fig, ax = plt.subplots(figsize=(10, 6))
iv_series.dropna().plot(kind='barh', ax=ax, color='steelblue')
ax.axvline(x=0.02, color='red',    linestyle='--', label='Weak (0.02)')
ax.axvline(x=0.10, color='orange', linestyle='--', label='Medium (0.10)')
ax.axvline(x=0.30, color='green',  linestyle='--', label='Strong (0.30)')
ax.set_title('Information Value (IV) by Feature')
ax.set_xlabel('IV')
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Feature Engineering

Create derived features from the raw columns — a core scorecard developer skill.

**Key ratios to always consider:**
- **Debt-to-income (DTI)**: repayment burden relative to income
- **Enquiry recency**: proportion of 12m enquiries in last 6m (credit hunger signal)
- **Arrears intensity**: arrears balance per active credit
- **Negative history rate**: negative marks per active credit

In [ ]:
df = df_raw.copy()

# ── Feature 1: Debt-to-Income ─────────────────────────────────────────
# Monthly repayment / monthly income
monthly_income = df['ingresos_smlv'] * 1_160_000  # approx SMLV in COP
df['dti'] = df['cuota_mensual'] / monthly_income.replace(0, np.nan)

# ── Feature 2: Enquiry Recency Ratio ──────────────────────────────────
# Proportion of 12m enquiries that occurred in the last 6m
df['query_recency_ratio'] = (
    df['queries_6m'] / df['queries_12m'].replace(0, np.nan)
).clip(upper=1.0)

# ── Feature 3: Arrears Intensity ─────────────────────────────────────
df['arrears_per_credit'] = (
    df['total_arrears_balance'] / df['active_credits'].replace(0, np.nan)
)

# ── Feature 4: Negative History Rate ────────────────────────────────
df['neg_history_rate'] = (
    df['hist_neg_12m'] / (df['active_credits'] + 1)  # +1 to avoid div by zero
)

# ── Feature 5: Score Gap ─────────────────────────────────────────────
# Difference between external experian score and internal platam score
# Large gap may indicate inconsistency or bureau data lag
if 'platam_score' in df.columns:
    df['score_gap'] = df['experian_score'] - df['platam_score']

new_features = ['dti', 'query_recency_ratio', 'arrears_per_credit', 'neg_history_rate']
if 'score_gap' in df.columns:
    new_features.append('score_gap')

print('Engineered features summary:')
print(df[new_features].describe().T[['mean', 'std', '50%', 'min', 'max']].to_string())

# IV for new features
print('\nIV for engineered features:')
for feat in new_features:
    try:
        s = bad_rate_by_band(df, feat, n_bins=5)
        print(f'  {feat:<25s}: IV = {s["iv"].sum():.4f}')
    except Exception as e:
        print(f'  {feat:<25s}: Error — {e}')

---
## 4. PSI — Population Stability Index

**One of the most important monitoring metrics for a Scorecard Developer.**

PSI measures how much a distribution has shifted between two time periods.
Thresholds: <0.10 stable | 0.10–0.25 minor shift | >0.25 rebuild required.

In [ ]:
def compute_psi(expected_arr, actual_arr, n_bins=10):
    """
    PSI = Sum over bins of: (Actual% - Expected%) * ln(Actual% / Expected%)
    Uses training percentile boundaries from expected_arr.
    """
    expected_arr = np.array(expected_arr)
    actual_arr   = np.array(actual_arr)

    # Define bin edges from the expected (training) distribution
    percentiles   = np.linspace(0, 100, n_bins + 1)
    breakpoints   = np.unique(np.percentile(expected_arr[~np.isnan(expected_arr)], percentiles))

    expected_counts, _ = np.histogram(expected_arr[~np.isnan(expected_arr)], bins=breakpoints)
    actual_counts, _   = np.histogram(actual_arr[~np.isnan(actual_arr)],   bins=breakpoints)

    # Avoid zeros
    expected_pct = (expected_counts / expected_counts.sum()).clip(1e-8)
    actual_pct   = (actual_counts   / actual_counts.sum()  ).clip(1e-8)

    psi_components = (actual_pct - expected_pct) * np.log(actual_pct / expected_pct)
    return psi_components.sum(), psi_components, breakpoints


# Simulate a distribution shift: split dataset chronologically
# First 70% = 'training era', last 30% = 'monitoring era'
split_idx  = int(len(df) * 0.70)
df_train   = df.iloc[:split_idx]
df_monitor = df.iloc[split_idx:]

print('=== PSI by Feature ===')
feature_cols = ['experian_score', 'queries_6m', 'dti', 'active_credits',
                'creditos_mora', 'ingresos_smlv']

psi_results = {}
for feat in feature_cols:
    if feat not in df.columns:
        continue
    psi_val, _, _ = compute_psi(
        df_train[feat].dropna().values,
        df_monitor[feat].dropna().values
    )
    psi_results[feat] = psi_val
    status = 'STABLE' if psi_val < 0.10 else ('MONITOR' if psi_val < 0.25 else 'REBUILD')
    print(f'  {feat:<25s}: PSI = {psi_val:.4f}  [{status}]')

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['green' if v < 0.10 else ('orange' if v < 0.25 else 'red') for v in psi_results.values()]
ax.bar(psi_results.keys(), psi_results.values(), color=colors)
ax.axhline(0.10, color='orange', linestyle='--', label='0.10 threshold')
ax.axhline(0.25, color='red',    linestyle='--', label='0.25 threshold')
ax.set_title('Feature PSI (Population Stability Index)')
ax.set_ylabel('PSI')
plt.xticks(rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Model Training & Core Metrics

Reproduce the core pipeline in ~40 lines. Understand every parameter.

**Parameters to explain in an interview:**
- `scale_pos_weight`: handles class imbalance. = neg/pos count ratio
- `StratifiedKFold`: preserves bad rate in every fold
- `roc_auc` scoring: threshold-independent, good for imbalanced data
- Gini = 2×AUC−1: primary credit scoring discrimination metric

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# ── Prepare features ─────────────────────────────────────────────────
FEATURES = [
    'experian_score', 'queries_6m', 'queries_12m', 'active_credits',
    'creditos_mora', 'total_debt', 'total_arrears_balance',
    'l_principal', 'l_term', 'edad', 'ingresos_smlv',
    'cuota_mensual', 'hist_neg_12m', 'dti', 'query_recency_ratio',
    'arrears_per_credit', 'neg_history_rate',
    'departamento', 'genero'
]
FEATURES = [f for f in FEATURES if f in df.columns]
TARGET   = 'target'

df_model = df[FEATURES + [TARGET]].copy()

# Handle nulls (median for numeric, '__NULL__' for categorical)
num_cols = df_model[FEATURES].select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in FEATURES if c not in num_cols]

for c in num_cols:
    df_model[c] = df_model[c].fillna(df_model[c].median())
for c in cat_cols:
    df_model[c] = df_model[c].fillna('__NULL__')

# Bin numeric columns (5 quantile bins)
binner = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile', subsample=None)
df_model[num_cols] = binner.fit_transform(df_model[num_cols])

# OHE everything (after binning, numeric bins become categorical-like)
all_feat = num_cols + cat_cols
df_model[all_feat] = df_model[all_feat].astype(str)
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_ohe = ohe.fit_transform(df_model[all_feat])
X = pd.DataFrame(X_ohe, columns=ohe.get_feature_names_out(all_feat))
y = df_model[TARGET].values

print(f'Feature matrix shape: {X.shape}')
print(f'Target balance: {y.mean():.2%} bad rate')

In [ ]:
# ── Train / test split ────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# ── scale_pos_weight — explaining class imbalance handling ────────────
neg, pos = np.bincount(y_train.astype(int))
spw = neg / pos
print(f'Train set: {neg} goods, {pos} bads')
print(f'scale_pos_weight = {spw:.2f}  (up-weights bads by this factor)')

# ── XGBoost with 5-fold stratified CV ────────────────────────────────
xgb = XGBClassifier(
    scale_pos_weight=spw,
    eval_metric='logloss',
    random_state=42,
    verbosity=0
)

# Quick grid for demo (full grid in train_pipeline.py)
param_grid = {
    'n_estimators':  [100, 200],
    'max_depth':     [3, 5],
    'learning_rate': [0.05, 0.10],
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
gs  = GridSearchCV(
    xgb, param_grid,
    cv=skf, scoring='roc_auc',
    n_jobs=-1, refit=True
)
gs.fit(X_train, y_train)

best = gs.best_estimator_
y_prob = best.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

auc   = roc_auc_score(y_test, y_prob)
gini  = 2 * auc - 1
f1    = f1_score(y_test, y_pred)
bal_acc = balanced_accuracy_score(y_test, y_pred)

print(f'\nDev Test Metrics:')
print(f'  AUC-ROC          : {auc:.4f}')
print(f'  Gini             : {gini:.4f}')
print(f'  F1 Score         : {f1:.4f}')
print(f'  Balanced Accuracy: {bal_acc:.4f}')
print(f'  Best params      : {gs.best_params_}')

---
## 6. Calibration Curve

**Interview concept:** A model can have good discrimination (high Gini) but poor calibration
(predicted PD of 20% but actual bad rate of 35%).
Calibration ensures the model output is interpretable as a true probability of default.

In [ ]:
# ── Raw (uncalibrated) vs calibrated PD ──────────────────────────────
prob_true_raw, prob_pred_raw = calibration_curve(y_test, y_prob, n_bins=10, strategy='quantile')

# Isotonic calibration
calib_model = CalibratedClassifierCV(best, cv='prefit', method='isotonic')
calib_model.fit(X_test, y_test)
y_prob_cal = calib_model.predict_proba(X_test)[:, 1]
prob_true_cal, prob_pred_cal = calibration_curve(y_test, y_prob_cal, n_bins=10, strategy='quantile')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Calibration plot
ax = axes[0]
ax.plot(prob_pred_raw, prob_true_raw, marker='o', label='Uncalibrated', color='steelblue')
ax.plot(prob_pred_cal, prob_true_cal, marker='s', label='Isotonic Calibrated', color='green')
ax.plot([0, 1], [0, 1], linestyle='--', color='grey', label='Perfect calibration')
ax.set_title('Calibration Curve\n(Predicted PD vs Actual Bad Rate)')
ax.set_xlabel('Mean Predicted PD')
ax.set_ylabel('Actual Bad Rate')
ax.legend()

# Score distribution by target class
ax = axes[1]
ax.hist(y_prob[y_test == 0], bins=30, alpha=0.6, label='Goods (0)', color='steelblue', density=True)
ax.hist(y_prob[y_test == 1], bins=30, alpha=0.6, label='Bads (1)',  color='red',       density=True)
ax.set_title('Score Distribution by Target Class')
ax.set_xlabel('Predicted PD')
ax.set_ylabel('Density')
ax.legend()

plt.tight_layout()
plt.show()

print(f'\nAUC after calibration (should be same): {roc_auc_score(y_test, y_prob_cal):.4f}')

---
## 7. Cut-off Strategy Table

**A key deliverable for this role.**

Given a calibrated score, show the business:
- At each possible cut-off: approval rate, expected bad rate, Precision, Recall
- This directly supports credit policy decisions

**Interview framing:** "I produced a cut-off strategy table showing approval rate vs bad rate
at 20 threshold points, enabling the credit committee to select the cut-off aligned with
their loss appetite."

In [ ]:
def cutoff_strategy_table(y_true, y_scores, thresholds=None):
    """Produce approval rate vs bad rate table at each threshold."""
    if thresholds is None:
        thresholds = np.arange(0.05, 0.95, 0.05)

    rows = []
    total = len(y_true)
    overall_bad_rate = y_true.mean()

    for t in thresholds:
        approved = y_scores < t           # approve if predicted PD below threshold
        n_approved = approved.sum()
        if n_approved == 0:
            continue
        n_bads_in_approved = y_true[approved].sum()
        approval_rate = n_approved / total
        bad_rate_approved = n_bads_in_approved / n_approved
        bad_rate_reduction = 1 - (bad_rate_approved / overall_bad_rate)

        rows.append({
            'cut_off':              round(t, 2),
            'approval_rate_%':      round(approval_rate * 100, 1),
            'n_approved':           n_approved,
            'bad_rate_%':           round(bad_rate_approved * 100, 2),
            'bad_rate_reduction_%': round(bad_rate_reduction * 100, 1),
        })

    return pd.DataFrame(rows)


strategy_tbl = cutoff_strategy_table(y_test, y_prob_cal)
print('=== Cut-off Strategy Table ===')
print(strategy_tbl.to_string(index=False))

# Plot
fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()
ax1.plot(strategy_tbl['cut_off'], strategy_tbl['approval_rate_%'], 'b-o', label='Approval Rate %')
ax2.plot(strategy_tbl['cut_off'], strategy_tbl['bad_rate_%'],      'r-s', label='Bad Rate %')
ax1.set_xlabel('Cut-off (Predicted PD Threshold)')
ax1.set_ylabel('Approval Rate (%)', color='blue')
ax2.set_ylabel('Bad Rate of Approved (%)', color='red')
ax1.set_title('Cut-off Strategy: Approval Rate vs Bad Rate Trade-off')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.show()

---
## 8. Score Decile Table

**The standard credit scoring performance table.**
Shows the model's ability to rank-order risk across the portfolio.
Decile 1 = highest risk (highest PD), Decile 10 = lowest risk.

In [ ]:
def score_decile_table(y_true, y_scores):
    """Build score decile analysis table."""
    df_d = pd.DataFrame({'y': y_true, 'score': y_scores})
    df_d['decile'] = pd.qcut(df_d['score'], q=10, labels=False, duplicates='drop')
    df_d['decile'] = df_d['decile'] + 1   # 1-indexed

    total_bads  = df_d['y'].sum()
    total_goods = len(df_d) - total_bads

    result = []
    for dec in sorted(df_d['decile'].unique()):
        sub = df_d[df_d['decile'] == dec]
        bads  = sub['y'].sum()
        goods = len(sub) - bads
        total = len(sub)
        result.append({
            'decile':         dec,
            'min_score':      round(sub['score'].min(), 3),
            'max_score':      round(sub['score'].max(), 3),
            'total':          total,
            'bads':           int(bads),
            'goods':          int(goods),
            'bad_rate_%':     round(100 * bads / total, 2),
            'pct_bads_%':     round(100 * bads / total_bads, 2),
            'pct_goods_%':    round(100 * goods / total_goods, 2),
        })

    tbl = pd.DataFrame(result)
    tbl['cum_pct_bads_%']  = tbl['pct_bads_%'].cumsum()
    tbl['cum_pct_goods_%'] = tbl['pct_goods_%'].cumsum()
    tbl['ks'] = (tbl['cum_pct_bads_%'] - tbl['cum_pct_goods_%']).abs()
    return tbl


decile_tbl = score_decile_table(y_test, y_prob)
print('=== Score Decile Table ===')
print(decile_tbl.to_string(index=False))
print(f"\nKS Statistic: {decile_tbl['ks'].max():.2f}%")

# ROC Curve with Gini annotation
fpr, tpr, _ = roc_curve(y_test, y_prob)
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC (Gini = {gini:.3f})')
ax.plot([0, 1], [0, 1], linestyle='--', color='grey')
ax.fill_between(fpr, tpr, alpha=0.1, color='steelblue')
ax.set_xlabel('False Positive Rate (1 - Specificity)')
ax.set_ylabel('True Positive Rate (Sensitivity)')
ax.set_title('ROC Curve')
ax.legend()
plt.tight_layout()
plt.show()

---
## 9. Score PSI — Model Monitoring

After training, compute PSI between the training score distribution and the test score distribution.
In production you would compute this monthly between training population and current population.

In [ ]:
train_scores = best.predict_proba(X_train)[:, 1]
test_scores  = best.predict_proba(X_test)[:, 1]

psi_score, psi_comps, breakpoints = compute_psi(train_scores, test_scores, n_bins=10)
print(f'Score PSI (train vs test): {psi_score:.4f}')
status = 'STABLE' if psi_score < 0.10 else ('MONITOR' if psi_score < 0.25 else 'REBUILD')
print(f'Status: {status}')

# Distribution comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(train_scores, bins=30, alpha=0.6, label='Training', density=True, color='steelblue')
ax.hist(test_scores,  bins=30, alpha=0.6, label='Test',     density=True, color='orange')
ax.set_title(f'Score Distribution Shift\nPSI = {psi_score:.4f} [{status}]')
ax.set_xlabel('Predicted PD')
ax.set_ylabel('Density')
ax.legend()

ax = axes[1]
bin_labels = [f'{b:.2f}' for b in breakpoints[:-1]]
ax.bar(range(len(psi_comps)), psi_comps,
       color=['red' if c > 0.025 else 'steelblue' for c in psi_comps])
ax.set_title('PSI Components by Bin')
ax.set_xlabel('Score Bin')
ax.set_ylabel('PSI Component')
ax.axhline(0, color='grey', linewidth=0.5)

plt.tight_layout()
plt.show()

---
## 10. Feature Importance — Gain vs Permutation

**Interview concept:** Gain-based importance (XGBoost native) can be misleading for high-cardinality
features. Permutation importance is model-agnostic and more reliable.
The pipeline cross-validates both — features with high gain but low permutation importance
are flagged as suspicious (possible overfit or data leakage).

In [ ]:
from sklearn.inspection import permutation_importance

# Gain-based importance
gain_imp = pd.Series(best.feature_importances_, index=X_train.columns).sort_values(ascending=False)

# Permutation importance on test set
perm_result = permutation_importance(
    best, X_test, y_test,
    n_repeats=5, random_state=42, scoring='roc_auc', n_jobs=-1
)
perm_imp = pd.Series(perm_result.importances_mean, index=X_test.columns).sort_values(ascending=False)

# Top 20 by gain
top20 = gain_imp.head(20).index

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

gain_imp[top20].sort_values().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 20 Features — Gain Importance')
axes[0].set_xlabel('Gain')

perm_imp[top20].sort_values().plot(kind='barh', ax=axes[1], color='orange')
axes[1].axvline(x=0.001, color='red', linestyle='--', label='Threshold 0.001')
axes[1].set_title('Top 20 Features — Permutation Importance')
axes[1].set_xlabel('Mean AUC Drop')
axes[1].legend()

plt.tight_layout()
plt.show()

# Suspicious features: high gain but near-zero permutation importance
gain_cutoff = gain_imp.quantile(0.50)
suspicious = [
    f for f in gain_imp.index
    if gain_imp[f] >= gain_cutoff and perm_imp.get(f, 0) < 0.001
]
print(f'Suspicious features (high gain, low perm): {suspicious}')

---
## 11. DuckDB SQL Practice

Run real SQL on the CSV using DuckDB — no database setup needed.
This is the fastest way to practice the SQL patterns from the guide.

**Install:** `pip install duckdb`

In [ ]:
try:
    import duckdb

    con = duckdb.connect()

    # Register the DataFrame as a SQL table
    con.register('credit_applications', df_raw)

    # --- Query 1: Bad rate by department ---
    q1 = """
    SELECT
        departamento,
        COUNT(*)                                  AS total_applications,
        SUM(target)                               AS total_bads,
        ROUND(AVG(target) * 100, 2)               AS bad_rate_pct,
        ROUND(AVG(experian_score), 1)             AS avg_experian_score
    FROM credit_applications
    GROUP BY departamento
    ORDER BY bad_rate_pct DESC
    """
    print('=== Bad Rate by Department ===')
    print(con.execute(q1).df().to_string(index=False))

    # --- Query 2: Score decile analysis ---
    q2 = """
    WITH scored AS (
        SELECT
            cedula,
            experian_score,
            target,
            NTILE(10) OVER (ORDER BY experian_score DESC) AS decile
        FROM credit_applications
        WHERE experian_score IS NOT NULL
    )
    SELECT
        decile,
        COUNT(*)                                    AS applications,
        SUM(target)                                 AS bads,
        ROUND(100.0 * SUM(target) / COUNT(*), 2)   AS bad_rate_pct,
        ROUND(MIN(experian_score), 0)              AS min_score,
        ROUND(MAX(experian_score), 0)              AS max_score
    FROM scored
    GROUP BY decile
    ORDER BY decile
    """
    print('\n=== Experian Score Decile Table ===')
    print(con.execute(q2).df().to_string(index=False))

except ImportError:
    print('DuckDB not installed. Run: pip install duckdb')
    print('Showing pandas equivalent instead...')

    # Pandas equivalent of Query 1
    result = (df_raw.groupby('departamento')
              .agg(total=('target','count'), bads=('target','sum'),
                   avg_score=('experian_score','mean'))
              .assign(bad_rate_pct=lambda x: (x['bads']/x['total']*100).round(2))
              .sort_values('bad_rate_pct', ascending=False))
    print(result.to_string())

---
## 12. Interview Cheat Sheet

Key numbers and facts to remember about this specific pipeline:

In [ ]:
print('=== PIPELINE PARAMETER CHEAT SHEET ===')
print()
print('HOLDOUT_PCT        = 0.15  → 15% held out before any preprocessing')
print('NULL_THRESHOLD_PCT = 40    → columns >40% null are dropped')
print('VARIATION_THRESHOLD= 0.02  → KS/CramersV < 0.02 → flagged low-info')
print('N_BINS             = 5     → quantile binning of numeric features')
print('CV_FOLDS           = 5     → StratifiedKFold cross-validation')
print('SCORING            = roc_auc → threshold-independent, good for imbalance')
print('scale_pos_weight   = neg/pos → up-weights minority (bad) class')
print('GAIN_TOP_PCT       = 50    → top 50% gain features cross-checked with perm importance')
print('PERM_MIN_THRESHOLD = 0.001 → perm importance < 0.001 = suspicious')
print()
print('=== METRIC CHEAT SHEET ===')
print()
print('Gini = 2×AUC − 1    |  >0.30 acceptable, >0.45 strong')
print('KS                  |  >20% good separation')
print('PSI (score)         |  <0.10 stable | 0.10-0.25 monitor | >0.25 rebuild')
print('IV                  |  <0.02 useless | >0.50 suspect leakage')
print('Brier Score         |  lower = better calibration')
print()
print('=== DATASET CHEAT SHEET ===')
print(f'Shape              : {df_raw.shape}')
print(f'Bad rate           : {df_raw["target"].mean():.2%}')
print(f'Numeric features   : {len(num_cols)}')
print(f'Categorical feature: {len(cat_cols)}')
print(f'Post-OHE features  : {X.shape[1]}')